## 1. Exploratory analysis

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv('../data/orders_medium.csv')
df.head()

,order_id,customer_id,restaurant_id,order_time,delivery_time,status
0,O00001,C1234,R041,2023-01-17,2023-01-17 00:43:00,Late
1,O00002,C1017,R019,2023-04-24,2023-04-24 00:33:00,Cancelled
2,O00003,C0488,R045,2024-01-17,2024-01-17 00:29:00,Cancelled
3,O00004,C1452,R086,2023-03-27,2023-03-27 00:31:00,Late
4,O00005,C0915,R002,2024-01-09,2024-01-09 01:02:00,Cancelled


## 2. Ingestion Test

Test ingestion to raw table using db connection

In [10]:
from sqlalchemy import create_engine
import os

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)


In [12]:
df["source_file"] = "orders_medium.csv"
df.head()

,order_id,customer_id,restaurant_id,order_time,delivery_time,status,source_file
0,O00001,C1234,R041,2023-01-17,2023-01-17 00:43:00,Late,orders_medium.csv
1,O00002,C1017,R019,2023-04-24,2023-04-24 00:33:00,Cancelled,orders_medium.csv
2,O00003,C0488,R045,2024-01-17,2024-01-17 00:29:00,Cancelled,orders_medium.csv
3,O00004,C1452,R086,2023-03-27,2023-03-27 00:31:00,Late,orders_medium.csv
4,O00005,C0915,R002,2024-01-09,2024-01-09 01:02:00,Cancelled,orders_medium.csv


In [14]:
df.to_sql(
  'orders_medium', 
  engine, 
  schema='raw', 
  if_exists='replace', 
  index=False # Do not include index column in the database table
)

1000

## 3. Transformation test

Test transformation query

### 1. Connect to database

In [15]:
from pathlib import Path
import sys 

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-09-15 09:28:12,947 | INFO | db_connection | Database environment variables validated successfully.
2026-09-15 09:28:12,966 | INFO | db_connection | Building database URL for host=p1_new_technologies_postgres, port=5432, database=restaurant_management, user=postgres
2026-09-15 09:28:12,973 | INFO | db_connection | Creating SQLAlchemy engine.


### 2. SQL helper function 
(returns sql as pandas dataframe)

In [30]:
from sqlalchemy import text

def run_query(query: str) -> pd.DataFrame:
  with engine.connect() as conn:
    return pd.read_sql_query(text(query), conn)

### 3. Tranformation
Rules:
1. Validate `order_id`,`customer_id`and `restaurant_id`start with `O`,`C`and `R`respectively.
2. Cast `order_time`and `delivery_time` to `TIMESTAMP`

In [31]:
raw_order_medium_query = """
SELECT * FROM raw.orders_medium
"""

raw_order_medium_df = run_query(raw_order_medium_query)
raw_order_medium_df.head()

,order_id,customer_id,restaurant_id,order_time,delivery_time,status,source_file
0,O00001,C1234,R041,2023-01-17,2023-01-17 00:43:00,Late,orders_medium.csv
1,O00002,C1017,R019,2023-04-24,2023-04-24 00:33:00,Cancelled,orders_medium.csv
2,O00003,C0488,R045,2024-01-17,2024-01-17 00:29:00,Cancelled,orders_medium.csv
3,O00004,C1452,R086,2023-03-27,2023-03-27 00:31:00,Late,orders_medium.csv
4,O00005,C0915,R002,2024-01-09,2024-01-09 01:02:00,Cancelled,orders_medium.csv


### Apply the transformation

In [32]:
transformed_order_medium_query = """
  SELECT
    TRIM(order_id) AS order_id,
    TRIM(customer_id) AS customer_id,
    TRIM(restaurant_id) AS restaurant_id,
    CAST(order_time AS TIMESTAMP) AS order_timestamp,
    CAST(delivery_time AS TIMESTAMP) AS delivery_timestamp,
    TRIM(status) AS status,
    source_file
  FROM raw.orders_medium
  WHERE order_id LIKE 'O%' 
    AND customer_id LIKE 'C%' 
    AND restaurant_id LIKE 'R%';
  
"""

orders_medium_df = run_query(transformed_order_medium_query)
orders_medium_df.head()

,order_id,customer_id,restaurant_id,order_timestamp,delivery_timestamp,status,source_file
0,O00001,C1234,R041,2023-01-17,2023-01-17 00:43:00,Late,orders_medium.csv
1,O00002,C1017,R019,2023-04-24,2023-04-24 00:33:00,Cancelled,orders_medium.csv
2,O00003,C0488,R045,2024-01-17,2024-01-17 00:29:00,Cancelled,orders_medium.csv
3,O00004,C1452,R086,2023-03-27,2023-03-27 00:31:00,Late,orders_medium.csv
4,O00005,C0915,R002,2024-01-09,2024-01-09 01:02:00,Cancelled,orders_medium.csv
